# File encryption using a password

In [1]:
import base64
import secrets
from pathlib import Path

from cryptography.fernet import Fernet
from cryptography.hazmat.primitives.kdf.scrypt import Scrypt

In [2]:
DATA_PATH = Path().resolve().parent.parent/"data"
# print(DATA_PATH)

## Sample data file

In [3]:
raw_file = DATA_PATH/"datasets"/"simplemaps_worldcities.csv"
raw_file.is_file()  # check if exists

True

In [4]:
# check first lines
with open(raw_file, "r") as rfile:
    lines = rfile.readlines()

lines[:5]

['"city","city_ascii","lat","lng","country","iso2","iso3","admin_name","capital","population","id"\n',
 '"Tokyo","Tokyo","35.6897","139.6922","Japan","JP","JPN","Tōkyō","primary","37977000","1392685764"\n',
 '"Jakarta","Jakarta","-6.2146","106.8451","Indonesia","ID","IDN","Jakarta","primary","34540000","1360771077"\n',
 '"Delhi","Delhi","28.6600","77.2300","India","IN","IND","Delhi","admin","29617000","1356872604"\n',
 '"Mumbai","Mumbai","18.9667","72.8333","India","IN","IND","Mahārāshtra","admin","23355000","1356226629"\n']

## Setup encryption

### generate the salt used for key derivation
- use python secrets module because is more secure for random data generation

In [5]:
def generate_salt(size=16):
    return secrets.token_bytes(size)

salt = generate_salt()
salt

b'\x1d\x10A\xce\xa7aw\xc3}\xc2\x16\xfa\xa6\xdan\x15'

In [6]:
def store_salt(salt):
    with open(DATA_PATH/"salt.salt", "wb") as salt_file:
        salt_file.write(salt)

store_salt(salt)

In [7]:
def load_salt():
    return open(DATA_PATH/"salt.salt", "rb").read()

load_salt()

b'\x1d\x10A\xce\xa7aw\xc3}\xc2\x16\xfa\xa6\xdan\x15'

### derive the key from the password using the passed salt
- Used a key derivation function from cryptography
- use recommended valous for n, r, p based on RFC 7914 

In [8]:
def derive_key(salt, password):
    kdf = Scrypt(salt=salt, length=32, n=2**14, r=8, p=1)
    return kdf.derive(password.encode())

derive_key(
    salt,
    "my_password"
)

b'\xb3\x81\xa0\xfd~ZB\xe6\xc3\x15\xc4V{\x9a\xbf\x80D\xb7\x13j\x1cS\x1bQ<K\xf3\xc8\x10\xff|\xa3'

### generate key from a password

In [9]:
def generate_key(password, salt_size=16, load_existing_salt=False, save_salt=True):
    if load_existing_salt:
        # load existing salt
        salt = load_salt()
    elif save_salt:
        # generate new salt and save it
        salt = generate_salt(salt_size)
        with open(DATA_PATH/"salt.salt", "wb") as salt_file:
            salt_file.write(salt)
    # generate the key from the salt and the password
    derived_key = derive_key(salt, password)
    # encode it using Base 64 and return it
    return base64.urlsafe_b64encode(derived_key)

key = generate_key("my_password")
key

b'wbmtjcfWoxlFRzj-GwUuzb32Whq0CAbCSfjPJuOZKYk='

### encrypt file

In [10]:
def encrypt(filename, key):
    with open(filename, "rb") as file:
        # read all file data
        file_data = file.read()

    # encrypt data
    f = Fernet(key)
    encrypted_data = f.encrypt(file_data)

    # write the encrypted file
    with open(filename, "wb") as file:
        file.write(encrypted_data)

### decrypt file

In [11]:
def decrypt(filename, key):
    with open(filename, "rb") as file:
        # read the encrypted data
        encrypted_data = file.read()
    
    # decrypt data
    f = Fernet(key)
    try:
        decrypted_data = f.decrypt(encrypted_data)
    except cryptography.fernet.InvalidToken:
        print("Invalid token, most likely the password is incorrect")
        return
    
    # write the original file
    with open(filename, "wb") as file:
        file.write(decrypted_data)